In [13]:
import os
import getpass

from dotenv import load_dotenv

# Read the API key and model from .env. Values in .env replace existing environment variables.
load_dotenv(override=True)
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-4o-mini")  # must return logprobs; swap on openrouter.ai/models if needed


# FLARE

### What is FLARE?

[FLARE](https://arxiv.org/pdf/2305.06983.pdf) stands for **Forward-Looking Active REtrieval** augmented generation. It was introduced in 2023 in the paper "Active Retrieval Augmented Generation" by Zhengbao Jiang et al. The original code is in [the paper's GitHub repo](https://github.com/jzbjyb/FLARE), and the LangChain version used here draws heavily on it.

**In plain English:** a language model writes an answer a piece at a time. When it becomes unsure about a word, it pauses, searches for documents about that word, and rewrites that part of the answer using what it found.

## Why FLARE Matters

- Gives a language model access to external documents while it writes.
- Targets hallucination, where a model states something wrong with confidence.
- Aims for answers that are more accurate and better grounded.

## How the Loop Works

1. Draft a chunk of the answer. No retrieval happens yet.
2. Check the probability of each token (its **logprob**). Tokens below `min_prob` are marked as uncertain.
3. If nothing is uncertain, keep the chunk and continue.
4. If something is uncertain, turn the uncertain phrase into a question, search the documents, and regenerate the chunk with the retrieved text as context.
5. Repeat until the model writes `FINISHED`.

![FLARE animation](https://raw.githubusercontent.com/jzbjyb/FLARE/main/res/flare.gif)

## How FLARE Differs from Traditional RAG

- Traditional RAG retrieves once, using the user's question, and then generates.
- FLARE decides **when and what to retrieve** while it generates, so one answer can trigger several searches.

## Key Parameters

1. **`max_completion_tokens`** (set on `llm`, 1024): the most tokens generated in one draft before the uncertainty check.
2. **`min_prob`** (0.7 in this notebook): tokens with a probability below this value are marked as uncertain. The LangChain default is 0.2. A higher value flags more tokens, which triggers more searches.
3. **`max_iter`** (not set here, so the library default of 10 applies): the most loop iterations allowed.

In [14]:
import arxiv
import pymupdf
import requests
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter

from langchain_classic.chains import FlareChain
from langchain_classic.chains.flare.prompts import PROMPT, QUESTION_GENERATOR_PROMPT

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.callbacks.manager import AsyncCallbackManagerForRetrieverRun, CallbackManagerForRetrieverRun

# We set this so we can see what exactly is going on
from langchain_core.globals import set_verbose

set_verbose(True)


## Building the Retriever

Before FLARE can look things up, it needs a searchable copy of the paper. This is the same pattern as in the earlier RAG notebook:

1. Download the FLARE paper from arXiv and extract its text with PyMuPDF.
2. Split the text into chunks of 256 characters, with a 16-character overlap, using `RecursiveCharacterTextSplitter`. Short chunks make each search return a focused passage.
3. Turn each chunk into an **embedding**, a list of numbers that represents its meaning, using an OpenRouter embedding model.
4. Store the embeddings in FAISS, a fast local search library, and wrap it as a retriever.

In [15]:
# instantiate llm
# logprobs=True asks for the log probabilities of each output token. FLARE uses them to find uncertain tokens.
llm = ChatOpenRouter(
    model=MODEL,
    max_completion_tokens=1024,  # cap on tokens per generation step
    model_kwargs={"logprobs": True},
)

#instantiate embeddings model
# langchain_openrouter only provides chat models, so we use OpenAIEmbeddings pointed at OpenRouter.
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)

# fetch docs from arxiv, in this case it's the FLARE paper.
# arxiv 4.x has no PDF download helper, so we download each PDF and read its text with PyMuPDF.
arxiv_client = arxiv.Client()
docs = []
for result in arxiv_client.results(arxiv.Search(id_list=["2305.06983"], max_results=2)):
    if result.pdf_url is None:
        continue
    response = requests.get(result.pdf_url, timeout=60)
    response.raise_for_status()
    with pymupdf.open(stream=response.content, filetype="pdf") as pdf:
        text = "".join(page.get_text() for page in pdf)
    docs.append(Document(
        page_content=text,
        metadata={
            "Entry ID": result.entry_id,
            "Published": str(result.published.date()),
            "Title": result.title,
            "Authors": ", ".join(a.name for a in result.authors),
        },
    ))

# instantiate text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=16)

# split the document into chunks
doc_chunks = text_splitter.split_documents(docs)

# create vector store consisting of embeddigs for document
vector_store = FAISS.from_documents(doc_chunks, embedding=embeddings)

vector_store_retriever = vector_store.as_retriever()


## Which Models Work with FLARE?

FLARE needs a model that can do two things:

1. **Return token logprobs.** Logprobs show the model's confidence in each token it writes. Without them, FLARE cannot find uncertain phrases. Support depends on both the model and the OpenRouter provider serving it, so check with Cell 6 rather than assuming.
2. **Follow instructions.** The prompts ask the model to write a question for a phrase and to end with `FINISHED`. Weaker models may ignore this, and the loop then runs until it hits `max_iter`.

Two cautions. Logprobs measure confidence, not truth, so a model can be confidently wrong and skip retrieval. Also, the question generator does not need logprobs, so a cheaper text model could handle it. This notebook uses one model for everything. Each search round adds model calls, so cost and latency grow with the number of uncertain phrases.

## Check That the Model Returns Logprobs

FLARE needs the model to report how confident it was about each token. Not every model, and not every OpenRouter provider serving it, supports this.

Cell 6 sends one short message and checks that logprobs come back.

- **If it passes,** the model is suitable and you can continue.
- **If it fails** with an error such as `Provider returned error`, set `MODEL` in `.env` to a model that supports logprobs. The notebook's default is `openai/gpt-4o-mini`. Then re-run the notebook from the top, because `.env` is read in the first cell.

In [16]:
# Check that the model returns logprobs before running FLARE.
try:
    check = llm.invoke("Say hi in one word.")
except Exception as e:
    raise RuntimeError(
        f"{MODEL!r} failed when logprobs were requested ({e}). Some OpenRouter providers "
        "do not support logprobs. Set MODEL in .env to a model that does, such as openai/gpt-4o-mini."
    ) from e
logprobs = check.response_metadata.get("logprobs")
assert logprobs and logprobs.get("content"), "No logprobs returned. Choose another MODEL in .env."
print(check.content)
print(f"{len(logprobs['content'])} tokens with logprobs")


Hello!
2 tokens with logprobs


## Set Up the FLARE Chain

FLARE uses four parts:

- **Question generator** (`QUESTION_GENERATOR_PROMPT | llm | StrOutputParser()`): turns an uncertain phrase into a search question. It only produces text, so it does not need logprobs.
- **Response chain** (`PROMPT | llm`): writes the answer. This is the part that needs logprobs, because FLARE reads the token probabilities from it.
- **Retriever**: finds document chunks for each question.
- **`min_prob=0.7`**: the threshold below which a token counts as uncertain.

The chain is built from these parts instead of with `FlareChain.from_llm`, because in the installed `langchain_classic` version `from_llm` only accepts `ChatOpenAI`. Our model is served through OpenRouter, so we pass the parts in directly.

In [17]:
# FlareChain.from_llm only accepts ChatOpenAI, so build the chain from its parts.
# The retriever is required by FlareChain.
flare = FlareChain(
    question_generator_chain=QUESTION_GENERATOR_PROMPT | llm | StrOutputParser(),
    response_chain=PROMPT | llm,
    retriever=vector_store_retriever,
    min_prob=0.7,
    verbose=True,
)


## Start Generation

The cells below call `flare.invoke(...)`. Because `set_verbose(True)` is on, the output shows the loop as it runs:

- **`Current Response`**: the text accepted so far. The first pass is empty, and the first draft is written with no retrieved documents. It therefore comes from the model's own memory.
- **`Generated Questions`**: the questions created from uncertain phrases. A search only happens when this line appears.
- **`Finished chain`**: the loop has ended. The final answer is in the `response` field of the output.

In [18]:
flare.invoke("What is FLARE?")



> Entering new FlareChain chain...
Current Response: 
Generated Questions: ['What can FLARE refer to in various contexts?', 'What are some common contexts in which the term "FLARE" is used?', 'What is the term used to describe a sudden eruption of increased brightness on the Sun, often associated with magnetic energy release in the field of astronomy?', 'What is a solar flare?', 'What type of device is used in emergencies or signaling situations and produces a very bright light or intense heat?', 'What is a device that produces a very bright light or intense heat, often used in emergencies or signaling situations?', 'What category can FLARE belong to that involves frameworks for developing applications?', 'What software framework is often associated with application development and is referred to as FLARE?', 'What term refers to items or skills in some video games that are named "flare"?', 'What do some video games name items or skills that can perform specific functions within the g

{'user_input': 'What is FLARE?',
 'response': ' FLARE, or Forward-Looking Active REtrieval Augmented Generation, is a framework designed to enhance language models (LMs) by effectively combining information retrieval with knowledge generation. The core idea behind FLARE is that LMs should retrieve information only when they lack the necessary knowledge to generate accurate responses. This minimizes the chances of hallucination or generating irrelevant content.\n\nFLARE operates in a two-step process: first, it generates content based on the initial user input; if the model encounters uncertainty in generating the next part, it retrieves relevant documents to inform the next steps of generation. This approach has demonstrated superior or competitive performance across several knowledge-intensive generation tasks when compared against baseline models. The implementation details and effectiveness of FLARE demonstrate its ability to enhance the reliability of language generation by incorpo

In [19]:
flare.invoke("What are the steps in a FLARE pipeline?")



> Entering new FlareChain chain...
Current Response: 
Generated Questions: ['What does the FLARE pipeline involve for effectively executing a data processing and analysis task?', 'What is the first key step in the FLARE pipeline that involves collecting and gathering data from various sources?', 'What is the process involved in the first step of the FLARE pipeline?', 'What is the purpose of collecting and gathering data from various sources in a FLARE pipeline?', 'What step in the FLARE pipeline involves cleaning and preparing the data for analysis, including handling missing values, normalization, and data transformation?', 'What is the process in the FLARE pipeline that involves cleaning and getting the data ready for analysis, including addressing issues like missing values?', 'What are some methods included in the data preprocessing step to prepare data for analysis?', "What is the process of developing new features or variables to improve a model's performance within the FLARE p

{'user_input': 'What are the steps in a FLARE pipeline?',
 'response': ' In a FLARE pipeline, the following steps are typically followed:\n\n1. **Initial Query Generation**: The pipeline starts with the user input, which is used to formulate an initial query. This query is then used to retrieve relevant documents or information.\n\n2. **First Sentence Generation**: Using the retrieved documents alongside the user input, the model generates the first sentence (\\(\\hat{s_1} = LM([D_x, x])\\)) to kickstart the generation process.\n\n3. **Iterative Sentence Generation**: The model iteratively generates subsequent sentences. For each new sentence, it predicts the next tokens based on the current generated context. If any tokens are uncertain, the model retrieves additional relevant documents to aid in the generation.\n\n4. **Forward-Looking Retrieval**: The process includes a mechanism for forward-looking retrieval, where predictions about the upcoming content are anticipated and utilized 

In [20]:
flare.invoke("What does a typical retrieval instruction look like in FLARE?")



> Entering new FlareChain chain...
Current Response: 
Generated Questions: ['What are the first two words in the acronym FLARE that describe its characteristics?', 'What does the acronym FLARE stand for in the context of information retrieval?', 'What is a key component of a typical retrieval instruction in FLARE?', 'What are the components that a typical retrieval instruction in FLARE may include?', 'What phrase can be used to describe an example of how to format a query in FLARE for retrieving data?', 'What is an example of a typical retrieval instruction format in FLARE?', 'What might be an example of a query in a typical retrieval instruction in FLARE?', 'What component in a typical retrieval instruction in FLARE is used to refine the search results by specifying criteria such as date range and categories?', 'What is an example of a filter parameter that specifies a time frame in retrieval instructions for FLARE?', 'What are some examples of categories that can be used in a retri

{'user_input': 'What does a typical retrieval instruction look like in FLARE?',
 'response': ' In FLARE, retrieval instructions typically involve the iterative generation of search queries that help retrieve relevant information when the language model (LM) is uncertain about its next output. These queries are integrated into the generative process and can take the form of specific phrases or questions that guide the retrieval system. For example, an instruction might prompt the LM to generate a query concerning a specific topic or to seek clarification on a detail, thereby facilitating targeted retrieval of additional context or information that can enhance the quality and relevance of subsequent text generation. The retrieved documents are then used to regenerate the next sentence, ensuring that the model has access to accurate and pertinent data. \n\n'}

## Reading the Outputs

Look at the verbose output from the three runs above.

- **Question generation drifts.** In the first run, the questions cover solar flares, flare guns and video game items. The question generator is asked about one uncertain word, "FLARE", and does not see the paper. It guesses from everyday meanings of the word.
- **Answers can fill gaps with general knowledge.** The second answer describes a generic machine-learning pipeline (collecting data, cleaning it, choosing a model). That is not how the paper defines FLARE. The verbose output does not show which chunks were retrieved, so this run alone does not tell us why the answer drifted.
- **A plausible answer is not a checked answer.** The third answer reads well, but treat each claim as a hypothesis until you check it against the paper.
- **Try changing `min_prob`.** At 0.7, many tokens count as uncertain, so the loop searches often. Set it lower, for example to the library default of 0.2, and compare how many `Generated Questions` lines appear. Re-running cells may also give different answers.